In [27]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import albumentations as A
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import yaml
import time

from argparse import Namespace
from torch.utils.data import DataLoader

from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping

from SAR import SARModel, SARDataModule

In [28]:
pl.seed_everything(42, workers=True)
np.random.seed(42)

Seed set to 42


In [29]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print("Device: ", device)

if use_cuda:
    print('__CUDNN VERSION:', torch.backends.cudnn.version())
    print('__Number CUDA Devices:', torch.cuda.device_count())
    print('__CUDA Device Name:',torch.cuda.get_device_name(0))
    print('__CUDA Device Total Memory [GB]:',torch.cuda.get_device_properties(0).total_memory/1e9)

Device:  cuda
__CUDNN VERSION: 8700
__Number CUDA Devices: 1
__CUDA Device Name: NVIDIA GeForce GTX 1050 Ti
__CUDA Device Total Memory [GB]: 4.294705152


In [30]:
# Path to project root folder
SUBGROUPS_DIR = '..'
DATA_DIR = os.path.join(SUBGROUPS_DIR, 'data')

MODELS_DIR = os.path.join(SUBGROUPS_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
LOGS_DIR = os.path.join(SUBGROUPS_DIR, 'logs')
os.makedirs(LOGS_DIR, exist_ok=True)

RESULTS_DIR = os.path.join(SUBGROUPS_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Local Folders
ROOT_DIR = os.getcwd()
INPUT_DIR = os.path.join(ROOT_DIR, 'input')
os.makedirs(INPUT_DIR, exist_ok=True)
IMAGES_DIR = os.path.join(INPUT_DIR, 'images')
os.makedirs(IMAGES_DIR, exist_ok=True)
MASKS_DIR = os.path.join(INPUT_DIR, 'masks')
os.makedirs(MASKS_DIR, exist_ok=True)
OUTPUT_DIR = os.path.join(ROOT_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [31]:
with open(rf"../configs/sample.yaml", 'r') as file:
    config = yaml.safe_load(file)

# Importing code for VV to RGB convertation from *.py file

In [32]:
import importlib.util

spec=importlib.util.spec_from_file_location("vv2rgb", f"vv2rgb/{config['vv2rgb_name']}.py")

vv2rgb = importlib.util.module_from_spec(spec)
spec.loader.exec_module(vv2rgb)
                
NODATA_COLOR = vv2rgb.NODATA_RGB
CONST_NORM = A.Normalize(mean=vv2rgb.NORM_MEAN, std=vv2rgb.NORM_STD, max_pixel_value=255.0, p=1.0)

def convert_db_norm(file_path, show_plot=False):
    vv = cv2.imread(file_path, cv2.IMREAD_UNCHANGED)
    rgb = vv2rgb.convert(vv)

    if show_plot is True:
        plt.imshow(rgb)
        plt.show()

    return rgb

In [33]:
dataset_df = pd.read_csv('split.csv')

In [34]:
NATIVE_RESOLUTION = 640

resolution_chain = [stage['image_size'] for stage in config['hparams']]
dataset_df_chain = [dataset_df.copy() for _ in resolution_chain]

for i in range(len(resolution_chain)):
    resolution = resolution_chain[i]
    target_df = dataset_df_chain[i]

    resolution_images_dir = os.path.join(IMAGES_DIR, str(resolution))
    resolution_masks_dir = os.path.join(MASKS_DIR, str(resolution))

    os.makedirs(resolution_images_dir, exist_ok=True)
    os.makedirs(resolution_masks_dir, exist_ok=True)

    for index, row in dataset_df.iterrows():
        file_path = os.path.join(row.dir_name, f"{row.image_name}_VV.tif")
        mask_path = os.path.join(row.dir_name, f"{row.image_name}_GT_LABELS.tif")

        if not (os.path.exists(file_path) and os.path.exists(mask_path)):
            continue

        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        vv = cv2.imread(file_path, cv2.IMREAD_UNCHANGED)
        
        if resolution != NATIVE_RESOLUTION:
            vv = cv2.resize(vv, (resolution, resolution))
            # max pooling of mask using dilatation
            scale = NATIVE_RESOLUTION // resolution
            mask = cv2.dilate(mask, np.ones((scale, scale)))
            init = scale // 2 
            mask = mask[init::scale, init::scale]
            # resize by neares to make sure of required dimentions
            mask = cv2.resize(mask, (resolution, resolution), interpolation = cv2.INTER_NEAREST)

        image = vv2rgb.convert(vv)

        image_save2_path = os.path.join(resolution_images_dir, f"{row.image_name}_IMAGE.jpg")
        mask_save2_path = os.path.join(resolution_masks_dir, f"{row.image_name}_MASK.tif")

        cv2.imwrite(image_save2_path, image)
        cv2.imwrite(mask_save2_path, mask)

        target_df.loc[index, "image_path"] = image_save2_path
        target_df.loc[index, "mask_path"] = mask_save2_path
        print(f"Image and mask files generated for {row.image_name}")

    target_df.dropna(inplace=True)

C:\Users\yevhe\AppData\Local\Temp\ipykernel_2748\3570944170.py:44: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'e:\WORK\iMermaid\CreodiasMigration\input\images\32\20180318_0001_IMAGE.jpg' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  target_df.loc[index, "image_path"] = image_save2_path
C:\Users\yevhe\AppData\Local\Temp\ipykernel_2748\3570944170.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'e:\WORK\iMermaid\CreodiasMigration\input\masks\32\20180318_0001_MASK.tif' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  target_df.loc[index, "mask_path"] = mask_save2_path


Image and mask files generated for 20180318_0001
Image and mask files generated for 20180409_2319
Image and mask files generated for 20180425_2250
Image and mask files generated for 20180507_2250
Image and mask files generated for 20180510_0010
Image and mask files generated for 20180517_0001
Image and mask files generated for 20180522_0010
Image and mask files generated for 20180529_0001
Image and mask files generated for 20180608_0018b
Image and mask files generated for 20180608_0018c
Image and mask files generated for 20180608_0018
Image and mask files generated for 20180612_2250
Image and mask files generated for 20180617_1022
Image and mask files generated for 20180622_0001
Image and mask files generated for 20180625_2328b
Image and mask files generated for 20180625_2328
Image and mask files generated for 20180628_2353
Image and mask files generated for 20180701_2242
Image and mask files generated for 20180702_0018
Image and mask files generated for 20180704_0001
Image and mask fi

C:\Users\yevhe\AppData\Local\Temp\ipykernel_2748\3570944170.py:44: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'e:\WORK\iMermaid\CreodiasMigration\input\images\64\20180318_0001_IMAGE.jpg' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  target_df.loc[index, "image_path"] = image_save2_path
C:\Users\yevhe\AppData\Local\Temp\ipykernel_2748\3570944170.py:45: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'e:\WORK\iMermaid\CreodiasMigration\input\masks\64\20180318_0001_MASK.tif' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  target_df.loc[index, "mask_path"] = mask_save2_path


Image and mask files generated for 20180625_2328b
Image and mask files generated for 20180625_2328
Image and mask files generated for 20180628_2353
Image and mask files generated for 20180701_2242
Image and mask files generated for 20180702_0018
Image and mask files generated for 20180704_0001
Image and mask files generated for 20180712_2335
Image and mask files generated for 20180713_2242
Image and mask files generated for 20180729_1737
Image and mask files generated for 20180803_2354
Image and mask files generated for 20180806_1048
Image and mask files generated for 20180812_0302
Image and mask files generated for 20180822_1508
Image and mask files generated for 20180827_0504
Image and mask files generated for 20180830_2242
Image and mask files generated for 20180902_0001
Image and mask files generated for 20180902_2305
Image and mask files generated for 20180903_1422
Image and mask files generated for 20180911_1048
Image and mask files generated for 20180914_0001
Image and mask file

In [35]:
class TempModelCheckpoint(ModelCheckpoint):
    def _save_checkpoint(self, trainer, filepath):
        # trainer.lightning_module.save_transformed_model = True
        super()._save_checkpoint(trainer, filepath)

def create_callbacks_and_logger(model_params):
    # S2-V2-640-REF-Linknet-timm-mobilenetv3_small_minimal_100-imagenet-dbscale-epoch=92-valid_loss=0.27.ckpt
    model_name_base = f"{model_params.data_version}-{model_params.image_size}-REF-{model_params.arch_name}-{model_params.encoder_name}-{model_params.encoder_weights}-{model_params.model_version}"

    checkpoint_callback = ModelCheckpoint(dirpath=MODELS_DIR,
                                      filename=rf'{model_name_base}-{{epoch}}-{{valid_loss:.2f}}',
                                      monitor='valid_loss',
                                      verbose=True,
                                      save_top_k=1,
                                      mode='min'
                                     )
    print(f"Checkpoint File Name Format: {checkpoint_callback.filename}")

    temporary_checkpoint_callback = TempModelCheckpoint(dirpath=MODELS_DIR,
                                        filename=rf'temp',
                                        monitor='valid_loss',
                                        verbose=True,
                                        save_top_k=1,
                                        mode='min'
                                        )
    print(f"Temporary Checkpoint File Name Format: {temporary_checkpoint_callback.filename}")

    lr_monitor_callback = LearningRateMonitor()
    early_stop_callback = EarlyStopping(monitor='valid_loss', patience=10, verbose=True, mode='min')

    os.makedirs(os.path.join(LOGS_DIR, model_name_base), exist_ok=True)
    logger = TensorBoardLogger(save_dir=LOGS_DIR, name=model_name_base)
    
    return [checkpoint_callback, temporary_checkpoint_callback, lr_monitor_callback, early_stop_callback], logger, model_name_base

In [36]:
def eval_model(trainer, model, datamodule, save_to):
    train_dataloader = DataLoader(datamodule.train_dataset, shuffle=True, batch_size=datamodule.hparams.batch_size, drop_last=True)

    train_metrics = trainer.validate(model, dataloaders=train_dataloader, verbose=False)
    valid_metrics = trainer.validate(model, datamodule=datamodule, verbose=False)
    test_metrics = trainer.test(model, datamodule=datamodule, verbose=False)

    prefixes = ['train', 'valid', 'test']
    metrics = [train_metrics, valid_metrics, test_metrics]

    dic = { "mode": [] }
    for i in range(3):
        prefix = prefixes[i]
        scores = metrics[i][0]

        dic["mode"].append(prefix)
        for entry in scores.keys():
            clipped = '_'.join(entry.split('_')[1:])
            if clipped not in dic.keys():
                dic[clipped] = []

            dic[clipped].append(scores[entry])

    df = pd.DataFrame.from_dict(dic)
    print(df)

    df.to_csv(save_to)


In [37]:
for train_for_resolution in range(len(resolution_chain)):
    hparams = config['hparams'][train_for_resolution]
    # INIT MODEL PARAMS
    model_params = dict(data_version=config['data_version'],
                        model_version=f"{config['vv2rgb_name']}-{config['model_version']}",
                        arch_name=config['arch_name'],
                        encoder_name=config['encoder_name'],
                        encoder_weights=config['encoder_weights'],
                        load_weights_from= None if train_for_resolution == 0 else os.path.join(MODELS_DIR, 'temp.ckpt'),
                        decoder_attention_type=None,
                        learning_rate=hparams['learning_rate'],
                        auto_lr_find=False,
                        max_epochs=hparams['max_epochs'],
                        image_size=hparams['image_size'],
                        batch_size=hparams['batch_size'],
                        accumulate_grad_batches=hparams['accumulate_grad_batches'],
                        auto_scale_batch_size=None, #'binsearch',
                        loss_function='DICE',
                        loss_alpha=None,
                        loss_beta=None,
                        loss_gamma=None,
                        in_channels=3,
                        input_shape=(resolution_chain[train_for_resolution], resolution_chain[train_for_resolution], 3),
                        out_classes=1)

    model_params = Namespace(**model_params)

    # CREATE TRAINER
    callbacks, logger, model_base_name = create_callbacks_and_logger(model_params)

    trainer = pl.Trainer(accelerator='gpu',
                        fast_dev_run=False,
                        max_epochs=model_params.max_epochs,
                        check_val_every_n_epoch=1,
                        logger=logger,
                        log_every_n_steps=20,
                        callbacks=callbacks,
                        accumulate_grad_batches=model_params.accumulate_grad_batches,
                        deterministic='warn')
    
    # CREATE MODEL
    dataset_split_df = dataset_df_chain[train_for_resolution].copy()

    sentinel1_data = SARDataModule(dataset_split_df, const_norm=CONST_NORM, nodata_color=NODATA_COLOR, image_size=model_params.image_size, batch_size=model_params.batch_size)
    sentinel1_data.setup()

    model = SARModel(model_params)

    print(f"Model Parameters: {model.hparams}")
    print(f"Learning Rate: {model.learning_rate}")

    if train_for_resolution > 0:
        os.remove(os.path.join(MODELS_DIR, 'temp.ckpt'))

    # TRAIN MODEL
    if model.hparams_.auto_lr_find is True or model.hparams_.auto_scale_batch_size is not None:
        trainer.tune(model, datamodule=sentinel1_data)

    start = time.time()
    trainer.fit(model, datamodule=sentinel1_data)
    end = time.time()

    time_used = end - start

    print(f"Path of best model checkpoint: {callbacks[0].best_model_path.split('/')[-1]}")
    print(f"GPU Memory Allocated: {torch.cuda.max_memory_allocated()}")
    print(f"GPU Memory Reserved : {torch.cuda.max_memory_reserved()}")

    base_path = os.path.join(RESULTS_DIR, model_base_name)
    times_path = base_path + ".txt"

    with open(times_path, 'a+') as f:
        f.write(f"Stage {train_for_resolution} ({hparams['image_size']}x{hparams['image_size']}) trained in {time_used} seconds!\n{time_used}\n")

    # EVALUATE MODEL
    eval_model(trainer, model, sentinel1_data, base_path + ".csv")

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Checkpoint File Name Format: S1-32-REF-Linknet-timm-mobilenetv3_small_minimal_100-imagenet-dB-staged-{epoch}-{valid_loss:.2f}
Temporary Checkpoint File Name Format: temp
Model Parameters: "hparams_": Namespace(data_version='S1', model_version='dB-staged', arch_name='Linknet', encoder_name='timm-mobilenetv3_small_minimal_100', encoder_weights='imagenet', load_weights_from=None, decoder_attention_type=None, learning_rate=0.01, auto_lr_find=False, max_epochs=10, image_size=32, batch_size=32, accumulate_grad_batches=1, auto_scale_batch_size=None, loss_function='DICE', loss_alpha=None, loss_beta=None, loss_gamma=None, in_channels=3, input_shape=(32, 32, 3), out_classes=1)
Learning Rate: 0.01



  | Name    | Type     | Params
-------------------------------------
0 | model   | Linknet  | 856 K 
1 | loss_fn | DiceLoss | 0     
-------------------------------------
856 K     Trainable params
0         Non-trainable params
856 K     Total params
3.426     Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


e:\WORK\iMermaid\CreodiasMigration\.conda\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Validation DataLoader 0: 100%|██████████| 2/2 [00:00<00:00, 33.11it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
e:\WORK\iMermaid\CreodiasMigration\.conda\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:441: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 10.10it/s]
    mode      loss  per_image_iou  dataset_iou  per_image_f1  dataset_f1
0  train  0.575246       0.244734     0.284171      0.350495    0.442575
1  valid  0.491802       0.264529     0.359937      0.371377    0.529344
2   test  0.507133       0.278730     0.346295      0.388541    0.514441


In [38]:
os.remove(os.path.join(MODELS_DIR, 'temp.ckpt'))